# Hands-on Projects — RAG

**Module:** 04 — RAG

Build an end-to-end toy RAG system, then branch into evaluation, hybrid, and multi-tenant tracks.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Assemble ingest→chunk→embed→retrieve→generate end to end
- Add citations and a refusal path
- Extend into hybrid retrieval and a mini eval harness
- Document failure cases you observe


## Project: End-to-End Toy RAG

**Definition.** A complete teaching system: small corpus, in-memory index, packer, grounded prompt, and citation-aware answer object.

**Why it matters.** Integration teaches what slides cannot—contracts between stages.

**How it works.** Implement modules as functions; keep numpy/stdlib embeddings for the toy; swap in API embedders later with YOUR_API_KEY.

**Intuition.** One thin vertical slice beats seven disconnected demos.

**Common pitfalls.**
- Scope creep into agents before baseline works
- No logged hits

**When to use.** Capstone for Module 04.

```mermaid
flowchart TB
  A[Corpus Markdown] --> B[Chunk]
  B --> C[Embed]
  C --> D[(Memory index)]
  E[Ask] --> F[Retrieve]
  D --> F
  F --> G[Pack + prompt]
  G --> H[Answer + cites]
```

### Project tracks

| Track | Goal |
|-------|------|
| A. Baseline | E2E toy RAG + citations |
| B. Hybrid | Add BM25 + RRF |
| C. Eval | 15-question harness |
| D. Multi-tenant | ACL filters |


In [ ]:
# Demo 1 — corpus + chunk
CORPUS = {
  "refund.md": "# Refunds\n\nRefunds within 60 days of purchase. Keep your receipt.",
  "ship.md": "# Shipping\n\nStandard shipping takes 3-5 business days.",
  "security.md": "# Security\n\nReset passwords from Account Settings → Security.",
}

def chunk_docs(corpus, size=20):
    chunks = []
    for name, text in corpus.items():
        words = text.split(); i=0; n=0
        while i < len(words):
            piece = " ".join(words[i:i+size])
            chunks.append({"id": f"{name}#{n}", "doc": name, "text": piece})
            i += size; n += 1
    return chunks
CHUNKS = chunk_docs(CORPUS)
print(len(CHUNKS), CHUNKS[0])


In [ ]:
# Demo 2 — embed + search
import numpy as np
vocab = sorted({w.lower() for c in CHUNKS for w in c["text"].split()})

def emb(text):
    t = text.lower().split()
    v = np.array([t.count(w) for w in vocab], float)
    return v / (np.linalg.norm(v) + 1e-9)

M = np.stack([emb(c["text"]) for c in CHUNKS])

def search(q, k=3):
    s = M @ emb(q)
    idx = np.argsort(-s)[:k]
    return [(CHUNKS[i], float(s[i])) for i in idx]

for c, sc in search("how long are refunds valid?"):
    print(f"{c['id']:16s} {sc:.3f}  {c['text'][:50]}")


In [ ]:
# Demo 3 — pack, prompt, refuse-or-answer
def pack(hits, budget=300):
    out, n = [], 0
    for c, sc in hits:
        block = f"[{c['id']}] {c['text']}"
        if n + len(block) > budget: break
        out.append(block); n += len(block)
    return out

def answer(question, hits, min_score=0.15):
    if not hits or hits[0][1] < min_score:
        return {"answer": "I do not know based on the knowledge base.", "citations": []}
    ctx = pack(hits)
    cites = [h[0]["id"] for h in hits[: len(ctx)]]
    # Stand-in generator (swap for API call with YOUR_API_KEY)
    text = f"Based on sources, regarding '{question}': see {', '.join(cites)}."
    return {"answer": text, "citations": cites, "context": ctx}

print(answer("refund window", search("refund window")))
print(answer("quantum cryptography pricing", search("quantum cryptography pricing")))


In [ ]:
# Demo 4 — optional API swap sketch
import json
YOUR_API_KEY = "YOUR_API_KEY"
hits = search("shipping time")
ctx = pack(hits)
req = {
  "model": "gpt-4.1-mini",
  "messages": [
    {"role": "system", "content": "Cite chunk ids. Refuse if evidence is weak."},
    {"role": "user", "content": "CONTEXT:\n" + "\n".join(ctx) + "\n\nQ: shipping time?"},
  ],
}
print(json.dumps(req, indent=2)[:600])
print("Authorization: Bearer", YOUR_API_KEY[:8] + "...")


In [ ]:
# Demo 5 — Track C mini eval harness
EVAL = [
    {"q": "refund window", "gold": ["refund.md#0"]},
    {"q": "shipping days", "gold": ["ship.md#0"]},
    {"q": "reset password", "gold": ["security.md#0"]},
]

def recall(gold, ranked_ids, k=3):
    return len(set(gold) & set(ranked_ids[:k])) / max(1, len(gold))

scores = []
for row in EVAL:
    ids = [c["id"] for c, _ in search(row["q"], k=3)]
    scores.append(recall(row["gold"], ids))
    print(row["q"], ids, "R@3=", scores[-1])
print("macro recall@3", sum(scores)/len(scores))


### Try it yourself — Project: End-to-End Toy RAG

1. Track A: Add two more markdown docs and ensure citations still work.
2. Track B: Add a toy BM25 list and fuse with RRF before packing.
3. Track C: Grow EVAL to 15 questions including two expected refusals.
4. Track D: Add tenant metadata and filter search results by tenant.


## Project hardening checklist

**Definition.** Operational checks that turn a demo into something you could show a teammate.

**Why it matters.** Most RAG demos die in integration: logging, eval, and access control.

**How it works.** Add traces per stage, a freeze of the eval set, and ACL tests.

**Intuition.** If you cannot explain a wrong answer from logs, it is not shippable.

**Common pitfalls.**
- Only manual clicking for QA
- Secrets in notebooks

**When to use.** Immediately after the vertical slice works.


In [ ]:
# Demo 1 — trace object
def traced_ask(q):
    hits = search(q)
    ans = answer(q, hits)
    return {"query": q, "hit_ids": [c["id"] for c,_ in hits], **ans}
print(traced_ask("refund window").keys())


In [ ]:
# Demo 2 — secret hygiene
import os
assert "sk-" not in open(__file__, encoding="utf-8").read() if False else True
key = os.getenv("OPENAI_API_KEY", "YOUR_API_KEY")
print("using", key[:8] + "...")


### Try it yourself — Project hardening checklist

1. Write a README-free checklist in comments for your team handoff.
2. Capture three failed queries and classify retrieval vs generation faults.


## Glossary

- **vertical slice**: End-to-end thin implementation
- **eval harness**: Fixed questions + metrics for regressions


### Workshop drill — Hands-on Projects — RAG (1)

Diagram the data flow on paper, then implement one missing log line per stage.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 1 — Hands-on Projects — RAG
stages = ['ingest','chunk','embed','retrieve','pack','generate']
for s in stages:
    print(f'log.{{s}}.ok = ?')


### Workshop drill — Hands-on Projects — RAG (2)

Create two adversarial queries (one ID-heavy, one paraphrase-heavy) and compare hits.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 2 — Hands-on Projects — RAG
queries = ['error code E42-refund', 'how do I get my money back?']
for q in queries:
    print('Q:', q)
    print('  TODO: print top-3 ids')


### Workshop drill — Hands-on Projects — RAG (3)

Write a refusal test: empty hits must not produce a confident numeric answer.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 3 — Hands-on Projects — RAG
def must_refuse(hits):
    return (not hits) or hits[0].get('score',0) < 0.2
assert must_refuse([])
assert must_refuse([{'score': 0.05}])
assert not must_refuse([{'score': 0.9}])
print('refusal tests ok')


### Workshop drill — Hands-on Projects — RAG (4)

Estimate cost: vary top_k and context tokens; print a small table.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 4 — Hands-on Projects — RAG
rows = []
for k in [2,4,8,16]:
    toks = k*400
    rows.append((k, toks, round(toks/1e6*0.5, 5)))
print('k  ctx_toks  approx_$')
for r in rows:
    print(*r)


### Workshop drill — Hands-on Projects — RAG (5)

Add one metadata field and filter it in retrieval (tenant or product).

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 5 — Hands-on Projects — RAG
docs = [{'id':'a','tenant':'acme'},{'id':'b','tenant':'beta'}]
tenant='acme'
print([d for d in docs if d['tenant']==tenant])


### Workshop drill — Hands-on Projects — RAG (6)

Write a gold eval row (question + gold chunk ids) and a recall@k helper.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 6 — Hands-on Projects — RAG
gold={'q':'refund window','gold_ids':['C1']}
def recall(gold_ids, ranked, k=3):
    return len(set(gold_ids)&set(ranked[:k]))/max(1,len(gold_ids))
print(recall(gold['gold_ids'], ['C9','C1','C2'], 3))


### Workshop drill — Hands-on Projects — RAG (7)

Sketch a hybrid fusion: BM25 ranks + vector ranks → RRF.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 7 — Hands-on Projects — RAG
def rrf(lists, k=60):
    scores={}
    for lst in lists:
        for r,d in enumerate(lst,1):
            scores[d]=scores.get(d,0)+1/(k+r)
    return sorted(scores, key=scores.get, reverse=True)
print(rrf([['A','B'],['B','C']]))


## Summary & Key Takeaways

- A thin E2E slice beats disconnected component demos
- Citations + refusals are part of the happy path
- Eval harnesses prevent silent regressions
- Hybrid, ACL, and time filters are natural next tracks

### Practice

Finish Track A and one of B/C/D; present failure cases you found.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
